Pydantic

In [22]:
import os
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

custom_llm = ChatOpenAI(
    model="minimax-local",
    api_key=os.getenv("CHAT_API_KEY"),
    base_url=os.getenv("CHAT_API_BASE_URL"),
    temperature=0,
    verbose=True
)


In [25]:
from pydantic import BaseModel, Field 

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    release_year: int = Field(..., description="The release year of the movie")
    ratings: float = Field(..., description="The ratings of the movie")

In [26]:
model_with_structure = custom_llm.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x791d14c50560>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x791d14b61790>, root_client=<openai.OpenAI object at 0x791d173746e0>, root_async_client=<openai.AsyncOpenAI object at 0x791d14b60380>, model_name='minimax-local', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://chat.soroco.com/api/v1', openai_proxy=None, stream_chunk_timeout=120.0), kwargs={'response_format': <class '__main__.Movie'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'release_year': {'description': 'T

In [4]:
resp = custom_llm.invoke("Provide details about the movie Inception")
resp

AIMessage(content='***Inception*** is a critically acclaimed science-fiction action film released in 2010, written and directed by **Christopher Nolan**. It is widely regarded as one of the most imaginative and complex films of the 21st century, blending a high-concept heist plot with deep emotional themes.\n\nHere are the detailed aspects of the movie:\n\n### 1. The Plot Concept\nThe film is centered around the concept of **"dream sharing."** In the world of the movie, technology exists that allows people to enter others\' dreams. \n\n*   **Extraction:** The protagonist, Dom Cobb, is an "extractor." He enters the subconscious of targets while they sleep to steal secrets (corporate espionage).\n*   **Inception:** The core conflict arises when Cobb is hired for the opposite task: instead of stealing an idea, he must **plant** one. This is "Inception." The goal is to convince a corporate heir to dissolve his father\'s empire, ensuring a fair market competition.\n\n### 2. The Mechanics of

In [27]:
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', director='Christopher Nolan', release_year=2010, ratings=-8.8)

### message output alongside parsed structure

In [6]:
from pydantic import BaseModel, Field 

class Movies(BaseModel):
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    release_year: int = Field(..., description="The release year of the movie")
    rating: float = Field(..., description="The movie's rating out of 10. It scales from 0 to 10, where 0 indicates a poor movie and 10 indicates an excellent movie.")
    
model_with_raw_structure = custom_llm.with_structured_output(Movies, include_raw=True)
response = model_with_raw_structure.invoke("Provide details about the movie Inception")

In [7]:
response

{'raw': AIMessage(content='{\n  "title": "Inception",\n  "director": "Christopher Nolan",\n  "release_year": 2010,\n  "rating":-10\n}', additional_kwargs={'parsed': Movies(title='Inception', director='Christopher Nolan', release_year=2010, rating=-10.0), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 20, 'total_tokens': 63, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'hosted_vllm/gemma-local', 'system_fingerprint': None, 'id': 'chatcmpl-957d524e724fa440', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ec067-63ea-76d1-9ba4-6d636ec93210-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 43, 'total_tokens': 63, 'input_token_details': {}, 'output_token_details': {}}),
 'parsed': Movies(title='Inception', director='Christopher Nolan', release_year=2010, rating=-10.0),
 'parsing_error': None}

In [8]:
### Nested Structured Output

from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    role: str = Field(..., description="The role of the actor in the movie")

class MoviieDetails(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    cast: list[Actor] = Field(..., description="List of actors in the movie")
    genre: str = Field(..., description="The genre of the movie")
    budget: int = Field(..., description="The budget of the movie in millions")
    
model_with_nested_structure = custom_llm.with_structured_output(MoviieDetails)
nested_response = model_with_nested_structure.invoke("Provide details about the movie Inception, including its cast, genre, and budget.")
nested_response

MoviieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Marion Cotillard', role='Ariadne'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genre='Sci-Fi / Action / Heist', budget=-160000000)

## TypedDict    

In [9]:
from typing import TypedDict, Annotated

class MovieDict(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    director: Annotated[str, ..., "The director of the movie"]
    release_year: Annotated[int, ..., "The release year of the movie"]
    ratings: Annotated[float, ..., "The movie's rating out of 10"]
    
    
custom_llm_with_typed_dict = custom_llm.with_structured_output(MovieDict)
typed_dict_response = custom_llm_with_typed_dict.invoke("Provide details about the movie avengers")
typed_dict_response

{'title': 'The Avengers',
 'director': 'Joss Whedon',
 'release_year': 2012,
 'ratings': -10.0}

In [10]:
class ActorDict(TypedDict):
    name: Annotated[str, ..., "The name of the actor"]
    role: Annotated[str, ..., "The role of the actor in the movie"]
class MovieDetailsDict(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The release year of the movie"]
    cast: Annotated[list[ActorDict], ..., "List of actors in the movie"]
    genre: Annotated[list[str], ..., "The genre of the movie"]
    budget: Annotated[int, ..., "The budget of the movie in millions"]
custom_llm_with_nested_typed_dict = custom_llm.with_structured_output(MovieDetailsDict)
nested_typed_dict_response = custom_llm_with_nested_typed_dict.invoke("Provide details about the movie Inception, including its cast, genre, and budget.")
nested_typed_dict_response

{'title': 'Inception',
 'year': 2010,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Marion Cotillard', 'role': 'Ariadne'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'},
  {'name': 'Michael Caine', 'role': 'Professor Stephen Miles'}],
 'genre': ['Sci-Fi', 'Action', 'Adventure'],
 'budget': -160000000}

In [15]:
custom_llm.profile

# DataClasses

In [28]:
from langchain.agents import create_agent
from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=custom_llm,
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]


ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [29]:
from langchain.agents import create_agent
from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # Append Gupt in the end of the name
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=custom_llm,
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [30]:
from langchain.agents import create_agent
from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # Append Gupt in the end of the name
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=custom_llm,
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567. Add Gupta in the end of name"}]
})

result["structured_response"]

ContactInfo(name='John Doe Gupta', email='john@example.com', phone='(555) 123-4567')